In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully")

#csv upload

print("\nUploading CSV file")
uploaded = files.upload()
csv_file = list(uploaded.keys())[0]  # Gets the filename you uploaded

print(f" File uploaded: {csv_file}")

master_df = pd.read_csv(csv_file)

print(f"\n Data loaded successfully!")
print(f"Dataset shape: {master_df.shape[0]} rows × {master_df.shape[1]} columns")
print(f"\nFirst few rows:")
print(master_df.head())


#remove spaces

# Remove spaces and special characters from column names
master_df.columns = master_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '')

print("\nColumn names cleaned")
print(f"Columns: {list(master_df.columns)[:10]}...")  # Shows first 10


#check missing values

print("\n Handling missing values")

# Replace "-" with NaN
master_df = master_df.replace('-', np.nan)
master_df = master_df.replace('', np.nan)

# Check missing values
missing_count = master_df.isnull().sum()
print(f"\nMissing values per column:")
print(missing_count[missing_count > 0])

# Fill numerical columns with MEDIAN
numerical_cols = master_df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if master_df[col].isnull().sum() > 0:
        median_val = master_df[col].median()
        master_df[col].fillna(median_val, inplace=True)

# Fill categorical columns with MODE
categorical_cols = master_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if master_df[col].isnull().sum() > 0:
        mode_val = master_df[col].mode()[0] if len(master_df[col].mode()) > 0 else 'Unknown'
        master_df[col].fillna(mode_val, inplace=True)

print(" Missing values filled with median (numbers) and mode (categories)")


#fix data types

print("\n Fixing data types")

# Convert date columns to datetime
date_cols = [col for col in master_df.columns if 'pruned' in col or 'date' in col]
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

# Convert numerical columns that might be stored as text
numerical_patterns = ['rainfall', 'yield', 'wet', 'sunshine', 'carbon', 'ph_value', 'plucking', 'target', 'extent', 'age']
for col in master_df.columns:
    if any(pattern in col.lower() for pattern in numerical_patterns):
        try:
            master_df[col] = pd.to_numeric(master_df[col], errors='coerce')
        except:
            pass

print(" Data types fixed")

#remove duplicates

print("\nRemoving duplicates")
initial_rows = len(master_df)
master_df = master_df.drop_duplicates()
removed_rows = initial_rows - len(master_df)
print(f"Removed {removed_rows} duplicate rows")


#remove unnecseary columns

# Keep only useful columns, remove business metrics
cols_to_drop = [col for col in master_df.columns if any(x in col.lower() for x in ['revenue', 'nsa', 'cop', 'profit', 'loss'])]

if cols_to_drop:
    master_df = master_df.drop(columns=cols_to_drop)
    print(f" Dropped {len(cols_to_drop)} unnecessary columns")


#summary

print("\n" + "="*50)
print("Data Preprocessing done")
print("="*50)

print(f"\nFinal Dataset:")
print(f"  Rows: {master_df.shape[0]}")
print(f"  Columns: {master_df.shape[1]}")
print(f"\nData Types:")
print(master_df.dtypes)

print(f"\nMissing Values After Cleaning:")
print(master_df.isnull().sum().sum(), "total missing values")

print(f"\nDataset Preview:")
print(master_df.head())


print("\n Saving cleaned data...")
master_df.to_csv('master_data_cleaned.csv', index=False)
files.download('master_data_cleaned.csv')

print(" Cleaned CSV downloaded as 'master_data_cleaned.csv'")
